In [8]:
import tkinter as tk
from tkinter import ttk
import time
import random

# --- CẤU HÌNH BÀI TOÁN TOÀN CỤC ---
BANGS = ["WA", "NT", "SA", "Q", "NSW", "V", "T"]

# Mối quan hệ logic cố định của đồ thị để thuật toán duyệt cây AND-OR
GIAP_RANH = {
    "WA": ["NT", "SA"],
    "NT": ["WA", "SA", "Q"],
    "SA": ["WA", "NT", "Q", "NSW", "V"],
    "Q": ["NT", "SA", "NSW"],
    "NSW": ["Q", "SA", "V"],
    "V": ["SA", "NSW", "T"],
    "T": ["V"]
}

MAU_SAC = ["Red", "Green", "Blue"]
MAU_HEX = {"Red": "#ff4d4d", "Green": "#2ecc71", "Blue": "#3498db", None: "#f2f2f2"}

class GiaoDienToMauAndOrSoLe:
    def __init__(self, cua_so):
        self.cua_so = cua_so
        self.cua_so.title("Map Coloring - Partial Edge AND-OR Search")

        self.phan_bo_mau = {bang: None for bang in BANGS}
        self.so_buoc = 0
        
        # Biến điều khiển luồng thuật toán
        self.dang_chay = False
        self.dang_tam_dung = False
        
        self.cac_vung_giao_dien = {}
        self.toa_do_hien_tai = {}

        # --- KHUNG TRÊN: ĐIỀU KHIỂN ---
        khung_tren = tk.Frame(cua_so)
        khung_tren.pack(pady=10, fill="x")

        self.nhan_trang_thai = tk.Label(
            khung_tren, text="Trạng thái: Sẵn sàng. Nhấn 'Đặt Lại' để sinh địa hình so le, đổi vị trí!", anchor="w", font=("Arial", 11)
        )
        self.nhan_trang_thai.pack(side="left", padx=10, fill="x", expand=True)

        self.btn_chay = tk.Button(khung_tren, text="Chạy AND-OR", command=self.bat_dau_thuat_toan, bg="#007acc", fg="white", font=("Arial", 10, "bold"))
        self.btn_chay.pack(side="right", padx=5)
        
        self.btn_dung = tk.Button(khung_tren, text="Dừng thuật toán", command=self.dao_trang_thai_dung, state="disabled", bg="#e74c3c", fg="white")
        self.btn_dung.pack(side="right", padx=5)
        
        tk.Button(khung_tren, text="Đặt Lại", command=self.dat_lai_va_sinh_dia_hinh).pack(side="right", padx=5)

        # --- KHUNG CHÍNH ---
        khung_chinh = tk.Frame(cua_so)
        khung_chinh.pack(padx=10, pady=10, fill="both", expand=True)

        # Cột trái: Canvas hiển thị bản đồ địa hình tự nhiên
        khung_trai = tk.Frame(khung_chinh)
        khung_trai.pack(side="left", anchor="n")

        self.canvas = tk.Canvas(khung_trai, width=440, height=480, bg="#eef2f5", relief="raised", borderwidth=2)
        self.canvas.pack()

        khung_quy_tac = tk.Frame(khung_trai, bg="#888", padx=10, pady=10)
        khung_quy_tac.pack(pady=15)
        tk.Label(khung_quy_tac, text="AND-OR SEARCH GOAL STATE", font=("Arial", 10, "bold"), bg="#888", fg="white").pack()
        self.txt_rule = tk.Label(khung_quy_tac, text="Địa hình tự nhiên (tiếp xúc ~30% cạnh).\nVị trí thay đổi ngẫu nhiên khi Đặt Lại.", font=("Arial", 9), bg="#888", fg="#ddd")
        self.txt_rule.pack()

        # Cột phải: Nhật ký log theo phong cách ┌─ Bước ──┐
        khung_phai = tk.Frame(khung_chinh, padx=10)
        khung_phai.pack(side="left", fill="both", expand=True)
        tk.Label(khung_phai, text="LỘ TRÌNH PHÂN RÃ AND-OR TRỰC QUAN (LOG)", font=("Arial", 10, "bold"), fg="#007acc").pack(anchor="w", pady=2)
        
        thanh_cuon = tk.Scrollbar(khung_phai)
        thanh_cuon.pack(side="right", fill="y")

        self.o_chu_log = tk.Text(
            khung_phai, width=45, height=25, font=("Courier", 10, "bold"),
            bg="#fdfdfd", yscrollcommand=thanh_cuon.set
        )
        self.o_chu_log.pack(side="left", fill="both", expand=True)
        thanh_cuon.config(command=self.o_chu_log.yview)
        
        # Sinh map ngẫu nhiên lần đầu
        self.dat_lai_va_sinh_dia_hinh()

    def sinh_ban_do_so_le_tu_nhien(self):
        """Sinh địa hình co giãn, lệch trục, tiếp xúc tối thiểu 30% cạnh để map tự nhiên hơn"""
        # Sinh ngẫu nhiên độ rộng/cao của các khối đất
        w_trai = random.randint(110, 140)
        w_giua = random.randint(110, 135)
        w_phai = random.randint(110, 135)
        
        h_hang1 = random.randint(100, 125)
        h_hang2 = random.randint(110, 135)
        h_hang3 = random.randint(75, 95)

        # Các biến lệch trục (offset) để các ô trượt lên xuống so le nhau
        dy_trai = random.randint(10, 35)
        dy_giua = random.randint(-15, 15)
        dy_phai = random.randint(15, 40)
        
        dx_sa = random.randint(-20, 20)
        dx_nsw = random.randint(10, 30)
        dx_v = random.randint(-25, 25)

        x_goc, y_goc = 40, 50

        # Khối 1 (Cột trái)
        x1_l, y1_l = x_goc, y_goc + dy_trai
        x2_l, y2_l = x1_l + w_trai, y1_l + h_hang1 + h_hang2

        # Khối 2 (Cột giữa - Hàng 1)
        x1_m1, y1_m1 = x2_l, y_goc + dy_giua
        x2_m1, y2_m1 = x1_m1 + w_giua, y1_m1 + h_hang1

        # Khối 3 (Cột phải - Hàng 1)
        x1_r1, y1_r1 = x2_m1, y_goc + dy_phai
        x2_r1, y2_r1 = x1_r1 + w_phai, y1_r1 + h_hang1 + 20

        # Khối 4 (Cột giữa - Hàng 2) - Giao dịch so le với khối 2
        x1_m2, y1_m2 = x2_l + dx_sa, y2_m1
        x2_m2, y2_m2 = x2_m1 + 10, y1_m2 + h_hang2

        # Khối 5 (Cột phải - Hàng 2)
        x1_r2, y1_r2 = x2_m2, y2_r1
        x2_r2, y2_r2 = x2_r1 - dx_nsw, y1_r2 + (y2_m2 - y1_r2) + 15

        # Khối 6 (Hàng đáy)
        x1_b, y1_b = x2_l + dx_v, y2_m2
        x2_b, y2_b = x2_r2, y1_b + h_hang3

        # Khối 7 (Đảo độc lập phía dưới)
        x1_t, y1_t = x1_b + random.randint(30, 70), y2_b + 15
        x2_t, y2_t = x1_t + random.randint(110, 160), y1_t + 45

        # Tập hợp danh sách tọa độ các đa giác hình học vuông cạnh phẳng đẹp
        vung_dat_phat_sinh = [
            [x1_l, y1_l, x2_l, y1_l, x2_l, y2_l, x1_l, y2_l],
            [x1_m1, y1_m1, x2_m1, y1_m1, x2_m1, y2_m1, x1_m1, y2_m1],
            [x1_r1, y1_r1, x2_r1, y1_r1, x2_r1, y2_r1, x1_r1, y2_r1],
            [x1_m2, y1_m2, x2_m2, y1_m2, x2_m2, y2_m2, x1_m2, y2_m2],
            [x1_r2, y1_r2, x2_r2, y1_r2, x2_r2, y2_r2, x1_r2, y2_r2],
            [x1_b, y1_b, x2_b, y1_b, x2_b, y2_b, x1_b, y2_b],
            [x1_t, y1_t, x2_t, y1_t, x2_t, y2_t, x1_t, y2_t]
        ]

        # XÁO TRỘN VỊ TRÍ: Đổi chỗ ngẫu nhiên các khối đất cho 7 Bang đại diện
        random.shuffle(vung_dat_phat_sinh)
        
        self.toa_do_hien_tai = {}
        for idx, bang in enumerate(BANGS):
            self.toa_do_hien_tai[bang] = vung_dat_phat_sinh[idx]

    def ve_ban_do(self):
        self.canvas.delete("all")
        self.cac_vung_giao_dien = {}

        for bang in BANGS:
            toa_do = self.toa_do_hien_tai[bang]
            mau_hien_tai = self.phan_bo_mau[bang]
            mau_nen = MAU_HEX[mau_hien_tai]
            
            poly_id = self.canvas.create_polygon(toa_do, fill=mau_nen, outline="#2c3e50", width=2)
            
            xs = toa_do[0::2]
            ys = toa_do[1::2]
            cx = sum(xs) / len(xs)
            cy = sum(ys) / len(ys)
            
            mau_chu = "white" if mau_hien_tai in ["Red", "Blue"] else "black"
            txt_id = self.canvas.create_text(cx, cy, text=bang, font=("Arial", 11, "bold"), fill=mau_chu)
            
            self.cac_vung_giao_dien[bang] = (poly_id, txt_id)

    def cap_nhat_mau_nhanh(self, bang):
        mau_hien_tai = self.phan_bo_mau[bang]
        mau_nen = MAU_HEX[mau_hien_tai]
        poly_id, txt_id = self.cac_vung_giao_dien[bang]
        
        self.canvas.itemconfig(poly_id, fill=mau_nen)
        mau_chu = "white" if mau_hien_tai in ["Red", "Blue"] else "black"
        self.canvas.itemconfig(txt_id, fill=mau_chu)
        self.cua_so.update()

    def kiem_tra_dung_va_tre(self):
        while self.dang_tam_dung:
            if not self.dang_chay:
                return False
            self.cua_so.update()
            time.sleep(0.1)
        if not self.dang_chay:
            return False
        return True

    def and_or_search_stochastic(self, danh_sach_cho):
        """THUẬT TOÁN AND-OR SEARCH TRÊN ĐỒ THỊ BÀI TOÁN"""
        if not danh_sach_cho:
            return True

        if not self.kiem_tra_dung_va_tre(): return False

        # Chọn bài toán con ngẫu nhiên để phân rã (Nút AND)
        danh_sach_xao_tron = list(danh_sach_cho)
        random.shuffle(danh_sach_xao_tron)
        bang_hien_tai = danh_sach_xao_tron.pop()
        
        # --- NÚT OR: Chọn thử các phương án màu ngẫu nhiên ---
        danh_sach_mau_ngau_nhien = list(MAU_SAC)
        random.shuffle(danh_sach_mau_ngau_nhien)

        for mau in danh_sach_mau_ngau_nhien:
            if not self.kiem_tra_dung_va_tre(): return False
            
            self.so_buoc += 1
            if self.kiem_tra_hop_le(bang_hien_tai, mau):
                self.phan_bo_mau[bang_hien_tai] = mau
                self.cap_nhat_mau_nhanh(bang_hien_tai)
                self._ghi_log_khoi(self.so_buoc, bang_hien_tai, "OR-NODE", f"Chọn thử màu {mau}")
                time.sleep(0.5)

                # --- NÚT AND: Bắt buộc tất cả các nhánh con phía dưới đều thành công ---
                self._ghi_mot_dong_log(f"-> Nhánh AND: Tiếp tục giải {len(danh_sach_xao_tron)} vùng con...")
                
                if self.and_or_search_stochastic(danh_sach_xao_tron):
                    return True

                if not self.kiem_tra_dung_va_tre(): return False

                # Quay lui (Backtracking) khi nhánh AND thất bại
                self._ghi_mot_dong_log(f"<- Nhánh AND thất bại tại vùng {bang_hien_tai}")
                self.phan_bo_mau[bang_hien_tai] = None
                self.cap_nhat_mau_nhanh(bang_hien_tai)
                self._ghi_log_khoi(self.so_buoc, bang_hien_tai, "BACKTRACK", "Quay lui chọn nhánh màu khác!")
                time.sleep(0.3)
            else:
                self._ghi_mot_dong_log(f"Xung đột ranh giới: {bang_hien_tai} trùng màu {mau}")

        return False

    def kiem_tra_hop_le(self, bang, mau):
        for hx in GIAP_RANH[bang]:
            if self.phan_bo_mau[hx] == mau:
                return False
        return True

    def dao_trang_thai_dung(self):
        if not self.dang_chay: return
        
        if self.dang_tam_dung:
            self.dang_tam_dung = False
            self.btn_dung.config(text="Dừng thuật toán", bg="#e74c3c")
            self.nhan_trang_thai.config(text="Thuật toán đang tiếp tục chạy...", fg="blue")
        else:
            self.dang_tam_dung = True
            self.btn_dung.config(text="Tiếp tục chạy", bg="#27ae60")
            self.nhan_trang_thai.config(text="Đã DỪNG thuật toán khẩn cấp! Nhấn nút để chạy tiếp.", fg="#d35400")

    def bat_dau_thuat_toan(self):
        if self.dang_chay: return
        self.dang_chay = True
        self.dang_tam_dung = False
        self.so_buoc = 0
        
        self.btn_chay.config(state="disabled")
        self.btn_dung.config(state="normal", text="Dừng thuật toán", bg="#e74c3c")
        
        self.o_chu_log.config(state="normal")
        self.o_chu_log.delete(1.0, tk.END)
        self.o_chu_log.config(state="disabled")
        
        self.nhan_trang_thai.config(text="AND-OR Search đang phân rã bài toán...", fg="blue")
        
        for b in BANGS:
            self.phan_bo_mau[b] = None
            self.cap_nhat_mau_nhanh(b)
            
        success = self.and_or_search_stochastic(list(BANGS))
        
        if not self.dang_chay:
            self.nhan_trang_thai.config(text="Đã dừng và hủy tiến trình thành công.", fg="black")
        elif success:
            self.nhan_trang_thai.config(text=f"Thành công! Tìm ra lời giải sau {self.so_buoc} bước toán.", fg="green")
        else:
            self.nhan_trang_thai.config(text="Thất bại: Bản đồ ngẫu nhiên này không có lời giải 3 màu.", fg="red")
            
        self.dang_chay = False
        self.dang_tam_dung = False
        self.btn_chay.config(state="normal")
        self.btn_dung.config(state="disabled", text="Dừng thuật toán", bg="#e74c3c")

    def dat_lai_va_sinh_dia_hinh(self):
        self.dang_chay = False
        self.dang_tam_dung = False
        self.phan_bo_mau = {bang: None for bang in BANGS}
        self.so_buoc = 0
        
        self.sinh_ban_do_so_le_tu_nhien()
        self.ve_ban_do()
        
        self.o_chu_log.config(state="normal")
        self.o_chu_log.delete(1.0, tk.END)
        self.o_chu_log.config(state="disabled")
        
        self.btn_chay.config(state="normal")
        self.btn_dung.config(state="disabled", text="Dừng thuật toán", bg="#e74c3c")
        self.nhan_trang_thai.config(text="Đã đổi Vị Trí + Tạo địa hình so le ngẫu nhiên mới!", fg="black")

    def _ghi_log_khoi(self, buoc, vung, hanh_dong, ket_qua):
        self.o_chu_log.config(state="normal")
        m = self.phan_bo_mau
        trang_thai_str = (
            f"│ WA:{m['WA'] or '?':<5} NT:{m['NT'] or '?':<5} SA:{m['SA'] or '?':<5} │\n"
            f"│ Q :{m['Q'] or '?':<5} NSW:{m['NSW'] or '?':<5} V :{m['V'] or '?':<5} │\n"
            f"│ T :{m['T'] or '?':<5} {'':19} │"
        )
        
        log_txt = (
            f"┌─ Bước {buoc:02d} ──────────────────────────┐\n"
            f"│ Vùng: {vung:<10} | Kiểu: {hanh_dong:<8} │\n"
            f"│ Kết quả: {ket_qua:<25} │\n"
            f"{trang_thai_str}\n"
            f"└──────────────────────────────────────┘\n"
            f"                    ▼                    \n"
        )
        self.o_chu_log.insert(tk.END, log_txt)
        self.o_chu_log.see(tk.END)
        self.o_chu_log.config(state="disabled")

    def _ghi_mot_dong_log(self, txt):
        self.o_chu_log.config(state="normal")
        self.o_chu_log.insert(tk.END, f"  > {txt}\n")
        self.o_chu_log.config(state="disabled")

if __name__ == "__main__":
    root = tk.Tk()
    app = GiaoDienToMauAndOrSoLe(root)
    root.mainloop()